# 06 — Direct Preference Optimization (DPO) Training
**Goal**: Train on execution-grounded `(chosen, rejected)` pairs from the SFT model (`β=0.1`, LR `5e-5`, 3 epochs). Preference tests now use the **same `normalize_tests` harness as PPO/eval**. Produces `./checkpoints/dpo/final` for RQ5 vs PPO-dense.

**Paper run**: use `train[:500]` (or larger) for preference collection — `train[:50]` is smoke only.

---


## Step 1: Environment Setup & Universal Path Resolution

In [ ]:
!pip install -q "trl<0.12.0" peft transformers datasets bitsandbytes accelerate

import sys, os, shutil

repo_root = os.path.abspath(os.getcwd())
GIT_SHA = ''
if os.path.isdir('/kaggle/working'):
    !rm -rf /kaggle/working/src
    !git clone -b junior-A https://github.com/Oin19/self-correction-llm-rl.git temp_repo
    !cp -r temp_repo/src /kaggle/working/src
    _sha = !git -C temp_repo rev-parse HEAD
    GIT_SHA = _sha[0] if _sha else ''
    print(f'Cloned junior-A @ {GIT_SHA}')
    !rm -rf temp_repo
    repo_root = '/kaggle/working'

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path initialized: {repo_root}")

!pip uninstall -y torchao

import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.training.dpo import make_preference_pairs, run_dpo_training, parse_apps_test_cases
from src.utils.test_cases import normalize_tests

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized! (normalize_tests harness active)")


## Step 2: Collect Preference Trajectory Pairs `(prompt, chosen, rejected)`

Runs $K=3$ debug rollouts. Tests come from `normalize_tests` (packed multi-test partial credit, same as PPO). AC → chosen; CE/RE/WA/TLE/MLE → rejected.

In [ ]:
!pip uninstall -y torchao

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
SFT_CHECKPOINT = "./checkpoints/sft/final"
PREF_SPLIT = "train[:500]"   # paper: 500+; smoke: train[:50]
SEED = 42

search_paths = [
    "./checkpoints/sft/final",
    "/kaggle/working/checkpoints/sft/final",
    "/kaggle/input"
]

model_path = None
for p in search_paths:
    if os.path.exists(p):
        if os.path.exists(os.path.join(p, 'adapter_config.json')):
            model_path = p
            break
        for root, dirs, files in os.walk(p):
            if 'adapter_config.json' in files and 'sft' in root.lower():
                model_path = root
                break
    if model_path:
        break

if not model_path:
    raise FileNotFoundError("SFT adapter checkpoint is required for DPO; refusing base-model fallback.")

print(f"Loading model from {model_path} for DPO pair collection in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(model_name=model_path, load_in_4bit=False, lora_r=16)

print(f"\nLoading APPS {PREF_SPLIT} for preference pair generation...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split=PREF_SPLIT)
# load_from_cache_file=False: avoid stale cache after normalize_tests changes
apps_clean = apps.filter(lambda x: len(normalize_tests(x)) > 0, load_from_cache_file=False)
print(f"Prepared {len(apps_clean)} problems with executable tests.")

print("Generating preference dataset...")
preference_data = make_preference_pairs(apps_clean, model, tokenizer, K=3, seed=SEED)
print(f"Total DPO preference pairs generated: {len(preference_data)}")
if len(preference_data) == 0:
    raise ValueError("No execution-grounded DPO preference pairs were generated; refusing to train on an empty or reference-fallback dataset.")


## Step 3: Run DPO Training

Uses PEFT reference-model trick (no duplicate ref weights). Seed + git SHA recorded in `dpo_metadata.json`.

In [ ]:
print("Starting DPO Training...")
dpo_trainer = run_dpo_training(
    model=model,
    tokenizer=tokenizer,
    preference_data=preference_data,
    output_dir="./checkpoints/dpo",
    beta=0.1,
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    seed=SEED,
    commit_sha=GIT_SHA if 'GIT_SHA' in dir() else '',
)

print("\nDPO Training completed successfully!")
print("Saved final DPO adapter checkpoint to ./checkpoints/dpo/final")


## Step 4: Checkpoint Verification & Inference Test
Verifies saved DPO adapter files, reloads weights, confirms LoRA config, and executes 3 APPS inference tests.

In [ ]:
import os
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint_dir = None
search_roots = ['./checkpoints/dpo/final', '/content/checkpoints/dpo/final', '/content/self-correction-llm-rl/checkpoints/dpo/final', '/kaggle/working/checkpoints/dpo/final', '/kaggle/working', '/kaggle/input']

for root in search_roots:
    if os.path.exists(root):
        if os.path.exists(os.path.join(root, 'adapter_config.json')):
            checkpoint_dir = root
            break
        for r, dirs, files in os.walk(root):
            if 'adapter_config.json' in files and 'dpo' in r.lower():
                checkpoint_dir = r
                break
    if checkpoint_dir:
        break

if not checkpoint_dir:
    checkpoint_dir = "./checkpoints/dpo/final"

print(f"=== 1. Inspecting DPO Checkpoint Files in {checkpoint_dir} ===")
if os.path.exists(checkpoint_dir):
    for fname in sorted(os.listdir(checkpoint_dir)):
        fpath = os.path.join(checkpoint_dir, fname)
        if os.path.isfile(fpath):
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"  - {fname}: {size_mb:.2f} MB")

    print("\n=== 2. Verifying DPO LoRA Configuration ===")
    config = PeftConfig.from_pretrained(checkpoint_dir)
    print(f"  - lora_alpha: {getattr(config, 'lora_alpha', 'N/A')}")
    print(f"  - r: {getattr(config, 'r', 'N/A')}")
    print(f"  - target_modules: {list(getattr(config, 'target_modules', []))}")
    print(f"  - peft_type: {getattr(config, 'peft_type', 'N/A')}")

    print("\n=== 3. Reloading Saved DPO Model & Adapter ===")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    reloaded_dpo = PeftModel.from_pretrained(base_model, checkpoint_dir)
    reloaded_dpo.eval()
    print("Reloaded DPO adapter model successfully!")

    print("\n=== 4. Running APPS Inference Check (3 Examples) ===")
    sample_prompts = [
        apps_clean[0]['question'],
        apps_clean[1]['question'],
        apps_clean[2]['question']
    ]

    for idx, p in enumerate(sample_prompts):
        prompt_text = f"### Problem:\n{p[:300]}\n\n### Solution:\n```python\n"
        inputs = tokenizer(prompt_text, return_tensors="pt").to(next(reloaded_dpo.parameters()).device)
        with torch.no_grad():
            out = reloaded_dpo.generate(**inputs, max_new_tokens=100, do_sample=False)
        gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"\n--- DPO Inference Sample {idx + 1} ---")
        print(gen_text[:250] + "...")

    print("\nCheckpoint verification completed successfully: adapter files, LoRA configuration, model reload, and inference generation verified.")
